# ДЗ-1. Каталог аниме-изображений: REST API → PostgreSQL

## Описание
Загружаем записи из `GET https://api.nekosapi.com/v4/images?rating=safe`
в `anime.images` в соответствии с настройками из `.env`. В таблицу попадают URL, теги, автор и цвета; сами изображения не скачиваются.


## Импорты


In [ ]:
# по заданию тут нужно больше, но я часть функционала вынесла в src/anime_etl.py
# чтобы не засорять тут

import json
import logging
import sys
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "docker-compose.yml").is_file()
        and (path / "src/anime_etl.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Откройте ноутбук внутри клонированного репозитория etl-anime.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from anime_etl import Settings, extract, load, make_session, transform, verify

## Конфигурация


In [2]:
settings = Settings.from_env(ROOT / ".env")
extracted_at = datetime.now(UTC)
run_dir = ROOT / "data" / "runs" / extracted_at.strftime("%Y%m%dT%H%M%S%fZ")
run_dir.mkdir(parents=True, exist_ok=False)

logger = logging.getLogger("anime_etl")
for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False
for handler in (
    logging.StreamHandler(sys.stdout),
    logging.FileHandler(run_dir / "etl.log", encoding="utf-8"),
):
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    logger.addHandler(handler)

print(f"Источник: {settings.api_base_url}/images; rating=safe")
print(f"Лимит: {settings.max_records}; размер страницы: {settings.page_size}")
print(f"Логи: {run_dir.relative_to(ROOT)}")

Источник: https://api.nekosapi.com/v4/images; rating=safe
Лимит: 500; размер страницы: 100
Логи: data/runs/20261121T151125995191Z


## Extract


In [3]:
with make_session(settings) as session:
    raw_records = extract(settings, run_dir / "raw", session)
print(f"Получено записей: {len(raw_records)}")
display(pd.DataFrame(raw_records).head(3))

2026-11-21 18:11:26,423 INFO Extract: offset=0, получено=100, доступно=16209
2026-11-21 18:11:27,109 INFO Extract: offset=100, получено=100, доступно=16209
2026-11-21 18:11:27,793 INFO Extract: offset=200, получено=100, доступно=16209
2026-11-21 18:11:28,464 INFO Extract: offset=300, получено=100, доступно=16209
2026-11-21 18:11:29,147 INFO Extract: offset=400, получено=100, доступно=16209
Получено записей: 500


,id,url,rating,color_dominant,color_palette,artist_name,tags,source_url
0,29813,https://cdn.nekosapi.com/nekos-api/images/orig...,safe,"[78, 127, 142]","[[236, 232, 233], [55, 107, 125], [126, 204, 2...",None,"[wet, girl]",None
1,253,https://cdn.nekosapi.com/nekos-api/images/orig...,safe,"[238, 225, 213]","[[159, 90, 66], [122, 43, 42], [243, 235, 226]...",None,"[brown_hair, catgirl, girl, kemonomimi]",None
2,256,https://cdn.nekosapi.com/nekos-api/images/orig...,safe,"[64, 59, 83]","[[225, 208, 202], [62, 56, 80], [132, 108, 129...",None,"[weapon, girl, black_hair]",None


## Transform
- `id` → `image_id`, `url` → `image_url`; целочисленный ID и время в UTC.
- Отбрасываем записи с невалидными ID/URL
- Удаляем дубли ID, нормализуем теги и пропуски автора/источника.
- Переводим доминирующий цвет RGB в HEX; убираем некорректные цвета палитры.


In [4]:
images = transform(raw_records, extracted_at)
print(f"Подготовлено уникальных записей: {len(images)}")
display(images.head(3))

2026-11-21 18:11:29,181 INFO Transform: вход=500, результат=500, отброшено/дубли=0
Подготовлено уникальных записей: 500


,image_id,image_url,rating,artist_name,source_url,tags,dominant_color_hex,color_palette,extracted_at
0,29813,https://cdn.nekosapi.com/nekos-api/images/orig...,safe,None,None,"[girl, wet]",#4e7f8e,"[[236, 232, 233], [55, 107, 125], [126, 204, 2...",2026-11-21 15:11:25.995191+00:00
1,253,https://cdn.nekosapi.com/nekos-api/images/orig...,safe,None,None,"[brown_hair, catgirl, girl, kemonomimi]",#eee1d5,"[[159, 90, 66], [122, 43, 42], [243, 235, 226]...",2026-11-21 15:11:25.995191+00:00
2,256,https://cdn.nekosapi.com/nekos-api/images/orig...,safe,None,None,"[black_hair, girl, weapon]",#403b53,"[[225, 208, 202], [62, 56, 80], [132, 108, 129...",2026-11-21 15:11:25.995191+00:00


## Load


In [5]:
inserted_count = load(images, settings)
print(f"Добавлено новых записей: {inserted_count}")

2026-11-21 18:11:29,359 INFO Load: вставлено=0, уже существовало=500
Добавлено новых записей: 0


## Проверка
Повторная загрузка


In [6]:
total_before, _ = verify(settings)
inserted_again = load(images, settings)
total_after, first_five = verify(settings)
assert inserted_again == 0, "Повторная загрузка создала новые строки"
assert total_before == total_after, "Число строк изменилось после повторной загрузки"

print(f"Извлечено: {len(raw_records)}")
print(f"После Transform: {len(images)}")
print(f"Вставлено за запуск: {inserted_count}")
print(f"Результат SELECT COUNT(*): {total_after}")
print(f"Повторная загрузка: {inserted_again} новых строк")
print("Идемпотентность: OK")
display(first_five)

_ = (run_dir / "summary.json").write_text(
    json.dumps(
        {
            "extracted": len(raw_records),
            "transformed": len(images),
            "inserted": inserted_count,
            "total": total_after,
            "inserted_again": inserted_again,
        },
        indent=2,
    ),
    encoding="utf-8",
)

2026-11-21 18:11:29,586 INFO Load: вставлено=0, уже существовало=500
Извлечено: 500
После Transform: 500
Вставлено за запуск: 0
Результат SELECT COUNT(*): 500
Повторная загрузка: 0 новых строк
Идемпотентность: OK


,image_id,rating,artist_name,tags,dominant_color_hex,image_url
0,253,safe,None,"[brown_hair, catgirl, girl, kemonomimi]",#eee1d5,https://cdn.nekosapi.com/nekos-api/images/orig...
1,256,safe,None,"[black_hair, girl, weapon]",#403b53,https://cdn.nekosapi.com/nekos-api/images/orig...
2,261,safe,None,"[brown_hair, girl, school_uniform]",#c59790,https://cdn.nekosapi.com/nekos-api/images/orig...
3,264,safe,None,"[black_hair, girl, skirt, tree]",#dec493,https://cdn.nekosapi.com/nekos-api/images/orig...
4,276,safe,None,"[red_hair, shorts]",#a2505e,https://cdn.nekosapi.com/nekos-api/images/orig...
